# Semantic Search: Find sentences that are similar to an asked question

This notebook looks into symmetric and asymmetric search methods. These methods are tested with questions that should give a reasonable answer and questions it should not know an answer to.

Our semantic search is an asymmetric search task. Our questions are likely to be shorter than the sentences prompt back to us. Which was, in case of `corpus_free_3600_250606.csv`, 24 words per sentence on average.

(See `explore_corpus.ipynb` for the average number of words per sentence and other descriptive statistical properties.)

It appears that a symmetric approach can be optimised for a asymmetric query by using a cross-encoder for the top_k number of sentences.

### Settings

In [ ]:
# Experiment_name
experiment_name = "free_1000_251013_pest_PD"

# Number of sentences returned. (For proposed search optimalisation of SBERT)
top_k = 35

### Initialisation

#### Import and functions

In [ ]:
# Import
import pickle
import pandas as pd
from sentence_transformers import SentenceTransformer
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers import util
import scipy.stats as stats
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Functions
def preprocess_query(query, embedding_model):
    """Natural query goes in, preprocessed (lowercasing, remove punctuations) and embedded query goes out."""
    
    # Pre-processing
    query = query.lower().strip('\[.*?\]')

    # Encode query
    query = embedding_model.encode(query)

    return query

def symmetric_query(query_embedding, corpus_embedding, corpus_df: pd.Series, top_k: int = 10):
    """df_corpus: Series containing sentences and an index that corresponds with the sentence_id (pd.Series)"""
    # semantic_search uses the 'exact nearest neighbor' algorithm
    # https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html#approximate-nearest-neighbor
    # symmetric semantic search
    result = util.semantic_search(
        query_embeddings= query_embedding,
        corpus_embeddings= corpus_embedding,
        top_k= top_k,
    )

    # Place results in a data frame
    df = pd.DataFrame(
        {
            'vector_id': [it['corpus_id'] for it in result[0]], # sentence_id of cos_sim with highest score.
            'cos_sim': [it['score'] for it in result[0]] # cos_sim score of retrieved sentence-query.
        }
    )

    # Merge with corpus data frame
    df = pd.merge(df, corpus_df, left_on= 'vector_id', right_index=True)

    return df

def quick_join(results, df_meta):
    """Quick and dirty function for joining relevant metadata."""
    results = results.join(
        df_meta[
                ['pmid', 'is_accepted', 'is_published', 'is_retracted', 'pub_year', 'cited_by_count', 'referenced_count']
            ].rename(
                columns= {'pmid': 'paper_name'}
        ).set_index('paper_name'),
        on= 'paper_name'
    )

    return results

def print_results(results):
    results = results.reset_index(drop=True)
    for i in range(results.shape[0]):
        try:
            print(f"cos_sim: {results.loc[i,'cos_sim']:.3f} | cross_score: {results.loc[i,'cross_scores']:.3f} | pmid: {results.loc[i,'paper_name']} | year: {results.loc[i,'pub_year']} | published: {results.loc[i,'is_published']} | retracted: {results.loc[i,'is_retracted']}")
        except KeyError:
            print(f"cos_sim: {results.loc[i,'cos_sim']:.3f} | cross_score: N/A | pmid: {results.loc[i,'paper_name']} | year: {results.loc[i,'pub_year']} | published: {results.loc[i,'is_published']} | retracted: {results.loc[i,'is_retracted']}")
        print(results.loc[i,'sentence_text'])
        print("~~~")

#### Load corpus, vectors and used transformer

In [ ]:
# File names
embedding_file = f"embedding_{experiment_name}.pickle"

raw_corpus_file = f"corpus_{experiment_name}.csv"
meta_file = f"meta_{experiment_name}.csv"
used_st_model = "NeuML/pubmedbert-base-embeddings"
cs_model = "cross-encoder/stsb-roberta-base"

In [ ]:
# Load raw corpus
df_corpus = pd.read_csv(f'../../data/corpus/{raw_corpus_file}')
df_corpus.head()

In [ ]:
# Load sentence embeddings
with open(f"../../data/vectors/{embedding_file}", 'rb') as handle:
    embeddings = pickle.load(handle)

In [ ]:
# Load used transformer
model_st = SentenceTransformer(used_st_model)
model_cs = CrossEncoder(cs_model)

In [ ]:
# Load metadata
df_meta = pd.read_csv(f'../../data/meta/{meta_file}', index_col=0)
df_meta.head()

In [ ]:
# For collecting STS scores
results_pos = pd.DataFrame(columns=['cos_sim', 'cross_scores'], dtype= float)
results_neg = pd.DataFrame(columns=['cos_sim', 'cross_scores'], dtype= float)

### Semantic search

Note that if the embedding contains more than one million rows a different search method is needed. `util.semantic_search` uses exact nearest neighbor which in that case would be time consuming. It is advised to use approximate nearest neighbor then instead.

https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html#approximate-nearest-neighbor

In [ ]:
# Check if our matrix contains more than 1.000.000 rows:
embeddings.shape

We have less then 1,000,000 rows, thus exact nearest neighbor is good to use.

SBERT proposed optimalisation by re-ranking with cross-encoders: https://github.com/UKPLab/sentence-transformers/tree/master/examples/sentence_transformer/applications/retrieve_rerank

In [ ]:
# Display more characters in printed data frames
pd.set_option('display.max_colwidth', 110)

### Positive controls

#### "Positive control" question: Is the risk of Parkinson's disease lower with old age?

In [ ]:
# Query
query = "Is the risk of Parkinson's disease lower with old age?"

results = quick_join(
    symmetric_query(
        query_embedding= preprocess_query(
            query,
            model_st
        ),
        corpus_embedding= embeddings,
        corpus_df= df_corpus[['sentence_text','paper_name']],
        top_k= top_k
    ),
    df_meta
)

results.head(10)

In [ ]:
# all results
print_results(results.head())

In [ ]:
# Optimise with cross-encoder
cross_inp = [[query, hit] for hit in results['sentence_text'].values]
cross_scores = model_cs.predict(cross_inp)

results['cross_scores'] = cross_scores

results = results.sort_values(by='cross_scores', ascending=False)
results.head(10)

In [ ]:
# all results
print_results(results.head())

In [ ]:
# Collect STS scores
results_pos = pd.concat(
    [results_pos, results[['cos_sim', 'cross_scores']]],
    ignore_index= True
)

#### "Positive control" question: Is the risk of Parkinson's disease higher at a young age?

In [ ]:
# Query
query = "Is the risk of Parkinson's disease higher at a young age?"

results = quick_join(
    symmetric_query(
        query_embedding= preprocess_query(
            query,
            model_st
        ),
        corpus_embedding= embeddings,
        corpus_df= df_corpus[['sentence_text','paper_name']],
        top_k= top_k
    ),
    df_meta
)

results.head(10)

In [ ]:
# all results
print_results(results.head())

In [ ]:
# Optimise with cross-encoder
cross_inp = [[query, hit] for hit in results['sentence_text'].values]
cross_scores = model_cs.predict(cross_inp)

results['cross_scores'] = cross_scores

results = results.sort_values(by='cross_scores', ascending=False)
results.head(10)

In [ ]:
# all results
print_results(results.head())

In [ ]:
# Collect STS scores
results_pos = pd.concat(
    [results_pos, results[['cos_sim', 'cross_scores']]],
    ignore_index= True
)

#### "Positive control" question: Is there a link between cellular organelles and Parkinson's disease?

In [ ]:
# Query 
query = "Is there a link between cellular organelles and Parkinson's disease?"

results = quick_join(
    symmetric_query(
        query_embedding= preprocess_query(
            query,
            model_st
        ),
        corpus_embedding= embeddings,
        corpus_df= df_corpus[['sentence_text','paper_name']],
        top_k= top_k
    ),
    df_meta
)

results.head(10)

In [ ]:
# all results
print_results(results.head())

In [ ]:
# Optimise with cross-encoder
cross_inp = [[query, hit] for hit in results['sentence_text'].values]
cross_scores = model_cs.predict(cross_inp)

results['cross_scores'] = cross_scores

results = results.sort_values(by='cross_scores', ascending=False)
results.head(10)

In [ ]:
# all results
print_results(results.head())

In [ ]:
# Collect STS scores
results_pos = pd.concat(
    [results_pos, results[['cos_sim', 'cross_scores']]],
    ignore_index= True
)

### Negative controls

#### "Negative control" question: How many people live in The Netherlands?

In [ ]:
# Query
query = "How many people live in The Netherlands?"

results = quick_join(
    symmetric_query(
        query_embedding= preprocess_query(
            query,
            model_st
        ),
        corpus_embedding= embeddings,
        corpus_df= df_corpus[['sentence_text','paper_name']],
        top_k= top_k
    ),
    df_meta
)

results.head(10)

In [ ]:
# all results
print_results(results.head())

In [ ]:
# Optimise with cross-encoder
cross_inp = [[query, hit] for hit in results['sentence_text'].values]
cross_scores = model_cs.predict(cross_inp)

results['cross_scores'] = cross_scores

results = results.sort_values(by='cross_scores', ascending=False)
results.head(10)

In [ ]:
# all results
print_results(results.head())

In [ ]:
# Collect STS scores
results_neg = pd.concat(
    [results_neg, results[['cos_sim', 'cross_scores']]],
    ignore_index= True
)

#### "Negative control" question: What causes high and low tides?

In [ ]:
# Query
query = "What causes high and low tides?"

results = quick_join(
    symmetric_query(
        query_embedding= preprocess_query(
            query,
            model_st
        ),
        corpus_embedding= embeddings,
        corpus_df= df_corpus[['sentence_text','paper_name']],
        top_k= top_k
    ),
    df_meta
)

results.head(10)

In [ ]:
# all results
print_results(results.head())

In [ ]:
# Optimise with cross-encoder
cross_inp = [[query, hit] for hit in results['sentence_text'].values]
cross_scores = model_cs.predict(cross_inp)

results['cross_scores'] = cross_scores

results = results.sort_values(by='cross_scores', ascending=False)
results.head(10)

In [ ]:
# all results
print_results(results.head())

In [ ]:
# Collect STS scores
results_neg = pd.concat(
    [results_neg, results[['cos_sim', 'cross_scores']]],
    ignore_index= True
)

#### "Negative control" question: What is the answer to the Great Question, of Life, the Universe and Everything? (Douglas Adams)

In [ ]:
# Query
# query = "How much is four divided by two"
query = "What is the answer to the Great Question, of Life, the Universe and Everything?"

results = quick_join(
    symmetric_query(
        query_embedding= preprocess_query(
            query,
            model_st
        ),
        corpus_embedding= embeddings,
        corpus_df= df_corpus[['sentence_text','paper_name']],
        top_k= top_k
    ),
    df_meta
)

results.head(10)

In [ ]:
# all results
print_results(results.head())

In [ ]:
# Optimise with cross-encoder
cross_inp = [[query, hit] for hit in results['sentence_text'].values]
cross_scores = model_cs.predict(cross_inp)

results['cross_scores'] = cross_scores

results = results.sort_values(by='cross_scores', ascending=False)
results.head(10)

In [ ]:
# all results
print_results(results.head())

In [ ]:
# Collect STS scores
results_neg = pd.concat(
    [results_neg, results[['cos_sim', 'cross_scores']]],
    ignore_index= True
)

### Range of positive and negative STS controls significant?

In [ ]:
# Combine the positive and negative controls
overview = pd.merge(   
        results_pos,
        results_neg,
        left_index= True,
        right_index= True,
        suffixes= ['_positive', '_negative']
)

overview.head()

#### Visualisations

In [ ]:
overview.iloc[:, [0, 2]].plot.density()
plt.title("Cosine similarity score distributions of (+) and (-) retrieved sentences")
plt.xlabel('Cosine similarity score')
plt.xticks(np.arange(0, 1.1, 0.2))
plt.show()

In [ ]:
overview.iloc[:, [0, 2]].plot.hist(bins=30, alpha=0.5)
plt.title("Cosine similarity score distributions of (+) and (-) retrieved sentences")
plt.xlabel('Cosine similarity score')
plt.xticks(np.arange(0, 1.1, 0.2))
plt.show()

In [ ]:
overview.iloc[:, [1, 3]].plot.density()
plt.title("Cross encoder score distributions of (+) and (-) retrieved sentences")
plt.xlabel("Cross encoder score")
plt.xticks(np.arange(0, 1.1, 0.2))
plt.show()

In [ ]:
overview.iloc[:, [1, 3]].plot.hist(bins=30, alpha=0.5)
plt.title("Cross encoder score distributions of (+) and (-) retrieved sentences")
plt.xlabel("Cross encoder score")
plt.xticks(np.arange(0, 1.1, 0.2))
plt.show()

#### Tests

The data is continues. I expect it not to be normal distributed, let's check:

In [ ]:
# Check if it follows a normal distribution (it does not)
for col in overview.columns:
    anderson_test = stats.anderson(overview[col], method= "interpolate")
    print(f"Anderson-Darling test on {col}:\np-value = {anderson_test[1]} | statistic = {anderson_test[0]}\n")

All columns are not normal distributed (pvalue < 0.05), except for `cross_scores_pos` (pvalue > 0.05).

Thus, for cross_scores we have two datasets we would like to compare, though one is normal distributed while the other is not. We will use the Mann-Whitney test given that it is more robust in handling non-normal distributed data.

In [ ]:
for col in [0, 1]:
    mannwhitney_test = stats.mannwhitneyu(overview.iloc[:, col], overview.iloc[:, col +2])
    print(f"Mann-Whitney test on {overview.iloc[:, col].name[:-9]}:\np-value = {mannwhitney_test[1]:.2e} | statistic = {mannwhitney_test[0]}\n")

### Questions from thesis

#### From thesis: "Explain how the use of pesticides induce Parkinson's disease in people that live near affected environments."

In [ ]:
# Query
query = "Explain how the use of pesticides induces Parkinson's disease in people that live near affected environments."

results = quick_join(
    symmetric_query(
        query_embedding= preprocess_query(
            query,
            model_st
        ),
        corpus_embedding= embeddings,
        corpus_df= df_corpus[['sentence_text','paper_name']],
        top_k= top_k
    ),
    df_meta
)

results.head(10)

In [ ]:
# all results
print_results(results.head())

In [ ]:
# Optimise with cross-encoder
cross_inp = [[query, hit] for hit in results['sentence_text'].values]
cross_scores = model_cs.predict(cross_inp)

results['cross_scores'] = cross_scores

results = results.sort_values(by='cross_scores', ascending=False)
results.head(10)

In [ ]:
# all results
print_results(results.head())

#### From thesis: "How do pesticides affects the human body and give rise to Parkinson's disease?"

In [ ]:
# Query
query = "How do pesticides affects the human body and give rise to Parkinson's disease?"

results = quick_join(
    symmetric_query(
        query_embedding= preprocess_query(
            query,
            model_st
        ),
        corpus_embedding= embeddings,
        corpus_df= df_corpus[['sentence_text','paper_name']],
        top_k= top_k
    ),
    df_meta
)

results.head(10)

In [ ]:
# all results
print_results(results.head())

In [ ]:
# Optimise with cross-encoder
cross_inp = [[query, hit] for hit in results['sentence_text'].values]
cross_scores = model_cs.predict(cross_inp)

results['cross_scores'] = cross_scores

results = results.sort_values(by='cross_scores', ascending=False)
results.head(10)

In [ ]:
# all results
print_results(results.head())

#### From thesis: "What is the mode of action of rotenone?"

In [ ]:
# Query
query = "What is the mode of action of rotenone?"

results = quick_join(
    symmetric_query(
        query_embedding= preprocess_query(
            query,
            model_st
        ),
        corpus_embedding= embeddings,
        corpus_df= df_corpus[['sentence_text','paper_name']],
        top_k= top_k
    ),
    df_meta
)

results.head(10)

In [ ]:
# all results
print_results(results.head())

In [ ]:
# Optimise with cross-encoder
cross_inp = [[query, hit] for hit in results['sentence_text'].values]
cross_scores = model_cs.predict(cross_inp)

results['cross_scores'] = cross_scores

results = results.sort_values(by='cross_scores', ascending=False)
results.head(10)

In [ ]:
# all results
print_results(results.head())

#### From thesis: "Which pesticide is best characterized?"

In [ ]:
# Query
query = "Which pesticide is best described in scientific literature?"

results = quick_join(
    symmetric_query(
        query_embedding= preprocess_query(
            query,
            model_st
        ),
        corpus_embedding= embeddings,
        corpus_df= df_corpus[['sentence_text','paper_name']],
        top_k= top_k
    ),
    df_meta
)

results.head(10)

In [ ]:
# all results
print_results(results.head())

In [ ]:
# Optimise with cross-encoder
cross_inp = [[query, hit] for hit in results['sentence_text'].values]
cross_scores = model_cs.predict(cross_inp)

results['cross_scores'] = cross_scores

results = results.sort_values(by='cross_scores', ascending=False)
results.head(10)

In [ ]:
# all results
print_results(results.head())

END